In [11]:
import requests
import time
import json
import csv
from pathlib import Path
from datetime import datetime, timezone

In [12]:
TARGET_DATE = "2026-05-03"
OUTPUT = f"data_processed/btc_5m_{TARGET_DATE}.csv"
DELAY = 10  # seconds between requests

# All 5-min window start timestamps for the target UTC day
date_start = int(datetime(2026, 5, 3,  0,  0, 0, tzinfo=timezone.utc).timestamp())
date_end   = int(datetime(2026, 5, 3, 23, 55, 0, tzinfo=timezone.utc).timestamp())
timestamps = range(date_start, date_end + 1, 300)
slugs = [f"btc-updown-5m-{ts}" for ts in timestamps]

print(f"{len(slugs)} windows to fetch")
print(f"ETA: ~{len(slugs) * DELAY // 60} min")

288 windows to fetch
ETA: ~48 min


In [13]:
Path(OUTPUT).parent.mkdir(exist_ok=True)

with open(OUTPUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["slug", "startDate", "endDate", "p_up", "p_down"])

    for i, slug in enumerate(slugs):
        resp = requests.get(
            "https://gamma-api.polymarket.com/events",
            params={"slug": slug}
        )
        data = resp.json()

        if not data:
            print(f"[{i+1:03d}/{len(slugs)}] {slug} — not found")
        else:
            market = data[0]["markets"][0]
            prices = json.loads(market["outcomePrices"])
            writer.writerow([
                market["slug"],
                market["startDate"],
                market["endDate"],
                prices[0],
                prices[1],
            ])
            f.flush()
            print(f"[{i+1:03d}/{len(slugs)}] {slug} — Up={prices[0]}  Down={prices[1]}")

        if i < len(slugs) - 1:
            time.sleep(DELAY)

print(f"\nDone. Saved to {OUTPUT}")

[001/288] btc-updown-5m-1777766400 — Up=1  Down=0
[002/288] btc-updown-5m-1777766700 — Up=0  Down=1
[003/288] btc-updown-5m-1777767000 — Up=0  Down=1
[004/288] btc-updown-5m-1777767300 — Up=1  Down=0
[005/288] btc-updown-5m-1777767600 — Up=1  Down=0
[006/288] btc-updown-5m-1777767900 — Up=1  Down=0
[007/288] btc-updown-5m-1777768200 — Up=1  Down=0
[008/288] btc-updown-5m-1777768500 — Up=1  Down=0
[009/288] btc-updown-5m-1777768800 — Up=0  Down=1
[010/288] btc-updown-5m-1777769100 — Up=0  Down=1
[011/288] btc-updown-5m-1777769400 — Up=0  Down=1
[012/288] btc-updown-5m-1777769700 — Up=1  Down=0
[013/288] btc-updown-5m-1777770000 — Up=0  Down=1
[014/288] btc-updown-5m-1777770300 — Up=0  Down=1
[015/288] btc-updown-5m-1777770600 — Up=1  Down=0
[016/288] btc-updown-5m-1777770900 — Up=0  Down=1
[017/288] btc-updown-5m-1777771200 — Up=0  Down=1
[018/288] btc-updown-5m-1777771500 — Up=1  Down=0
[019/288] btc-updown-5m-1777771800 — Up=0  Down=1
[020/288] btc-updown-5m-1777772100 — Up=0  Down=1


In [14]:
import pandas as pd

df = pd.read_csv(OUTPUT)
df["startDate"] = pd.to_datetime(df["startDate"])
df["endDate"]   = pd.to_datetime(df["endDate"])
df.head(10)

,slug,startDate,endDate,p_up,p_down
0,btc-updown-5m-1777766400,2026-05-02 00:09:18.006875+00:00,2026-05-03 00:05:00+00:00,1,0
1,btc-updown-5m-1777766700,2026-05-02 00:16:07.703427+00:00,2026-05-03 00:10:00+00:00,0,1
2,btc-updown-5m-1777767000,2026-05-02 00:18:38.418282+00:00,2026-05-03 00:15:00+00:00,0,1
3,btc-updown-5m-1777767300,2026-05-02 00:23:53.983605+00:00,2026-05-03 00:20:00+00:00,1,0
4,btc-updown-5m-1777767600,2026-05-02 00:28:39.434744+00:00,2026-05-03 00:25:00+00:00,1,0
5,btc-updown-5m-1777767900,2026-05-02 00:34:09.713598+00:00,2026-05-03 00:30:00+00:00,1,0
6,btc-updown-5m-1777768200,2026-05-02 00:38:41.164999+00:00,2026-05-03 00:35:00+00:00,1,0
7,btc-updown-5m-1777768500,2026-05-02 00:43:38.228084+00:00,2026-05-03 00:40:00+00:00,1,0
8,btc-updown-5m-1777768800,2026-05-02 00:49:10.226718+00:00,2026-05-03 00:45:00+00:00,0,1
9,btc-updown-5m-1777769100,2026-05-02 00:54:08.091213+00:00,2026-05-03 00:50:00+00:00,0,1
